In [0]:
from pyspark.sql import functions as F

sales_hourly = spark.read.table("sales_project_streaming.gld.hourly_regional_sales_agg")
exp_hourly = spark.read.table("sales_project_streaming.gld.hourly_regional_expenses_agg")

combined = (sales_hourly.alias("s").join(exp_hourly.alias("e"), on = ["window_start", "city"], how = "full_outer")
            .select(F.coalesce(F.col("s.window_start"), F.col("e.window_start")).alias("window_start"),
                    F.coalesce(F.col("s.window_end"), F.col("e.window_end")).alias("window_end"),
                    F.coalesce(F.col("s.city"), F.col("e.city")).alias("city"),
                    F.coalesce(F.col("s.total_revenue"), F.lit(0)).alias("total_revenue"),
                    F.coalesce(F.col("e.total_spent"), F.lit(0)).alias("total_spent"))
            )

display(combined)

In [0]:
%sql
select * from sales_project_streaming.gld.daily_regional_profit_agg;

In [0]:
data = [
    ('{"name": "Sai", "hobbies":["gym", "code"]}',)
    ]

schema = ("name", "hobby")
jschema = ["value"]

df = spark.createDataFrame(data, jschema)

display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType(), True),
    StructField("hobbies", ArrayType(StringType()), True)
])

df = df.withColumn("parsed", F.from_json("value", schema)).select("parsed.name", "parsed.hobbies")

display(df)

In [0]:
df = df.withColumn("hobby", F.explode_outer("hobbies"))

display(df)

In [0]:
%sql
select * from sales_project_streaming.gld.hourly_regional_profit_agg
where date>current_date()-2;

In [0]:
%sql
select * from sales_project_streaming.gld.daily_regional_profit_agg